Language detection inside the worker processes, so both language detection and tokenization run in parallel.

Key points:

The main process only streams and groups raw rows into chunks (no language filtering).

Each worker instantiates its own langid + tiktoken encoder, filters its chunk for English, tokenizes kept rows, saves a .npy chunk file, and returns a small verification sample (first up to 20 token ids + decoded text).

The main process logs each chunk's sample as soon as a worker returns it and then merges all .npy chunk files into one memmapped .npy.

In [1]:
#!/usr/bin/env python3
"""
Extract -> Filter -> Tokenize -> Save (binary) pipeline
-----------------------------------------------------

This module is a notebook-ready, production-oriented pipeline that:
  - Streams text rows from Parquet files using pyarrow.dataset.
  - Groups raw rows into chunks (by approximate bytes) to balance
    per-worker work.
  - Dispatches chunks to worker processes (joblib loky backend) where
    each worker: detects English rows, tokenizes with tiktoken, appends
    optional EOS tokens, and writes a compact raw binary `.bin` file
    (uint16 when possible, fallback to uint32).
  - The main process collects chunk files and merges them into a single
    memory-map-friendly `tokens_merged.bin` and a small JSON index
    (`tokens_merged.json`) containing dtype and total token count.

Design goals / rationale (summary):
  - Compact storage: use uint16 when possible to halve disk/RAM vs int32.
  - Fast IO for training: write/read raw binary and use np.memmap on load.
  - Atomic writes: workers write temp files and publish with os.replace.
  - Low peak RAM: merge in small blocks using memmap-backed reads/writes.
  - Resumability: existing chunk files are honored (skipped on re-run).

Usage (high-level):
  1. Configure INPUT_DIR and OUTPUT_DIR at top of file.
  2. Run `run_pipeline()` which performs the full flow.
  3. Training should load `tokens_merged.json` and `tokens_merged.bin`
     using the `np.memmap` dtype given in the JSON index.

Notes & portability:
  - By default the code writes native-endian raw binary. If you need
    to share the `tokens_merged.bin` between machines with different
    endianness, adapt writers/readers to use explicit little-endian
    dtype (e.g. `astype('<u2')`) and record that in the JSON index.

"""

from __future__ import annotations

import csv
import gc
import json
import logging
import os
import re
import time
import uuid
from pathlib import Path
from time import perf_counter
from typing import Iterator, List, Optional, Tuple

import numpy as np
from joblib import Parallel, delayed, parallel_backend
import pyarrow as pa
import pyarrow.dataset as ds
import langid
import tiktoken

# ----------------- Configuration (tweak to your environment) -----------------
INPUT_DIR = "datasets/4GB"                # directory containing parquet files
PARQUET_GLOB = "*.parquet"
OUTPUT_DIR = Path("outputs/extract_v12")
MERGED_BIN = OUTPUT_DIR / "tokens_merged.bin"
MERGED_INDEX = OUTPUT_DIR / "tokens_merged.json"

BATCH_ROWS = 4096                           # pyarrow.RecordBatch row count
TARGET_CHUNK_BYTES = 50_000_000            # approximate bytes per dispatched chunk

TOKENIZER_MODEL = "gpt2"                  # tiktoken model name
N_JOBS = max(1, min(os.cpu_count() or 1, 16))
PREFETCH_FACTOR = 4
PARALLEL_VERBOSE = 5

ASCII_THRESHOLD = 0.5
LANG_MODE = "skip"                        # language-filtering mode: 'skip'|'ascii'|'accuracy'

ADD_EOS = True                             # append EOS token after each kept row

PROFILE = True
PROFILE_CSV = OUTPUT_DIR / "tokenization_profile.csv"

MERGE_BLOCK_TOKENS = 1_000_000             # number of tokens copied per inner loop during merge

ATOMIC_SAVE_MAX_RETRIES = 8
ATOMIC_SAVE_RETRY_DELAY = 0.2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------- Logging -----------------
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
run_id = int(time.time())
log_file = LOG_DIR / f"extract_{run_id}.log"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)
file_handler = logging.FileHandler(str(log_file), mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S"))
logger.addHandler(file_handler)
logger.info("Starting run; writing logs to %s", log_file)

# ----------------- Helper utilities (documented) -----------------

def salvage_stale_temp_files(out_dir: Path) -> int:
    """
    Repair leftover temporary artifacts from previously-interrupted runs.

    Many worker writes are implemented as atomic temporary files followed
    by os.replace(). Occasionally earlier runs leave `*.bin.tmp` files
    behind (for example under network mounts or interrupted runs). This
    helper renames those back to their final form so the pipeline can
    resume without reprocessing the same data.

    Parameters
    ----------
    out_dir : Path
        Directory where chunk `.bin` files are written.

    Returns
    -------
    int
        Number of files salvaged / renamed.
    """
    repaired = 0
    for p in out_dir.glob("chunk_*.bin.tmp"):
        final = p.with_suffix("")  # drop the '.tmp' suffix
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug("Could not salvage %s: %s", p, e)
    if repaired:
        logger.info("Salvaged %d stale temp files in %s", repaired, out_dir)
    return repaired


def list_parquet_files(input_dir: str, pattern: str = "*.parquet") -> List[str]:
    """
    Return a sorted list of parquet file paths in `input_dir` matching `pattern`.

    Raises FileNotFoundError if the input directory does not exist. Sorting
    ensures deterministic processing order which helps reproducibility.
    """
    p = Path(input_dir)
    if not p.exists():
        raise FileNotFoundError(f"Input dir not found: {p}")
    return sorted(str(x) for x in p.glob(pattern))


def choose_string_column(dataset: ds.dataset) -> str:
    """
    Pick the best string column name from a pyarrow dataset schema.

    Preference order: 'text', 'content', 'body', otherwise fall back to
    the first string column found. This allows the pipeline to work with
    multiple different parquet schemas without manual configuration.
    """
    schema = dataset.schema
    str_columns = [f.name for f in schema if pa.types.is_string(f.type)]
    for candidate in ("text", "content", "body"):
        if candidate in str_columns:
            return candidate
    if not str_columns:
        raise RuntimeError("No string column in parquet schema(s)")
    return str_columns[0]


def is_english_text(text: str, mode: str = "accuracy", ascii_threshold: float = 0.5) -> bool:
    """
    Lightweight heuristic language filter used in-worker.

    Modes
    -----
    - 'skip' : accept everything (fastest, no lang checks).
    - 'ascii' or 'fast' : accept only if ascii fraction >= ascii_threshold.
    - 'accuracy' : try langid.classify(text) and fall back to ascii fraction.

    The ascii fraction fallback protects against langid failures on
    extremely short strings.
    """
    if mode is None:
        mode = "accuracy"
    mode = mode.lower().strip()
    if mode == "skip":
        return True
    if not text:
        return False

    def ascii_fraction(s: str) -> float:
        total = len(s)
        if total == 0:
            return 0.0
        ascii_count = sum(1 for c in s if ord(c) < 128)
        return ascii_count / total

    if mode in ("ascii", "fast"):
        return ascii_fraction(text) >= ascii_threshold

    if mode == "accuracy":
        try:
            lang, _score = langid.classify(text)
        except Exception:
            lang = None
        if lang == "en":
            return True
        return ascii_fraction(text) >= ascii_threshold

    raise ValueError(f"is_english_text: unknown mode '{mode}'. Choose 'skip', 'accuracy' or 'ascii'.")


# ----------------- Streaming / chunking (main process) -----------------

def iter_raw_text_chunks_from_parquets(
    input_dir: str,
    pattern: str = "*.parquet",
    batch_rows: int = 4096,
    target_chunk_bytes: int = 50_000_000,
) -> Iterator[Tuple[List[str], int]]:
    """
    Stream parquet rows and yield lists of raw text rows grouped into chunks.

    Why chunk? Sending many small rows to workers individually incurs
    overhead. Grouping into roughly TARGET_CHUNK_BYTES keeps each worker
    busy for a reasonable time while bounding per-task memory.

    Yields
    ------
    (chunk_texts, chunk_id)
      - chunk_texts: List[str] of original text rows
      - chunk_id: integer id (monotonically increasing)
    """
    file_list = list_parquet_files(input_dir, pattern)
    if not file_list:
        logger.warning("No parquet files found for pattern %s in %s", pattern, input_dir)
        return

    dataset = ds.dataset(file_list, format="parquet")
    chosen_col = choose_string_column(dataset)
    logger.info("Using string column '%s' for text extraction", chosen_col)

    scanner = dataset.scanner(batch_size=batch_rows)
    chunk_id = 0
    buffer_texts: List[str] = []
    buffer_bytes = 0

    for record_batch in scanner.to_batches():
        arr = record_batch[chosen_col]
        for raw in arr.to_pylist():
            if raw is None:
                continue
            text = str(raw).strip()
            if not text:
                continue
            # approximate byte size using UTF-8 encoding; this gives a stable
            # packing heuristic across different languages and characters.
            b = len(text.encode("utf-8"))

            # if adding this row would overflow target chunk size, yield the
            # current buffer as one chunk and start a new buffer.
            if buffer_texts and (buffer_bytes + b) >= target_chunk_bytes:
                yield buffer_texts, chunk_id
                chunk_id += 1
                buffer_texts = []
                buffer_bytes = 0

            # if a single row is larger than target size, emit it as its own chunk
            if b >= target_chunk_bytes:
                yield [text], chunk_id
                chunk_id += 1
                continue

            buffer_texts.append(text)
            buffer_bytes += b

    if buffer_texts:
        yield buffer_texts, chunk_id


# ----------------- Atomic raw write (worker) -----------------

def _safe_atomic_write_raw_bytes(data_bytes: bytes, out_path: Path,
                                 max_retries: int = ATOMIC_SAVE_MAX_RETRIES,
                                 retry_delay: float = ATOMIC_SAVE_RETRY_DELAY) -> None:
    """
    Atomically write raw bytes to `out_path` by writing to a uniquely
    named temporary file in the same directory and then using os.replace.

    Rationale:
      - os.replace is atomic on POSIX and Windows semantics are also
        handled via retries; this avoids partial files being visible to
        other processes (e.g. downstream merges or training loaders).
      - We fsync the file descriptor before replace to minimize
        chance of data loss on crash.
    """
    out_dir = out_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp_name = f"{out_path.stem}.{os.getpid()}.{uuid.uuid4().hex}.bin.tmp"
    tmp_path = out_dir / tmp_name

    # write + flush + fsync to make temp file robust to process crashes
    with open(tmp_path, "wb") as fh:
        fh.write(data_bytes)
        fh.flush()
        os.fsync(fh.fileno())

    if not tmp_path.exists():
        raise FileNotFoundError(f"Temp write failed: {tmp_path}")

    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            os.replace(str(tmp_path), str(out_path))
            return
        except (FileNotFoundError, PermissionError) as e:
            last_exc = e
            time.sleep(retry_delay)
    raise last_exc or RuntimeError("Atomic replace failed for unknown reason")


# ----------------- Worker: filter + tokenize + write .bin -----------------

def _get_eos_token_id(enc: "tiktoken.Encoding") -> Optional[int]:
    """
    Heuristically obtain the tokenizer's EOS token id if available.

    Some tokenizer APIs expose a special token directly; as a fallback
    we hardcode the common GPT-2 EOS id (50256).
    """
    try:
        try:
            ids = enc.encode("", allowed_special={"<|endoftext|>"})
            if ids:
                return int(ids[0])
        except Exception:
            pass
        try:
            ids = enc.encode("")
            if ids:
                return int(ids[0])
        except Exception:
            pass
    except Exception:
        pass

    try:
        if TOKENIZER_MODEL.lower().startswith("gpt2"):
            return 50256
    except Exception:
        pass
    return None


def tokenize_filter_and_save(
    chunk_texts: List[str],
    chunk_id: int,
    out_dir: str,
    tokenizer_model: str,
    sample_n: int = 20,
    lang_mode: str = LANG_MODE,
    ascii_threshold: float = ASCII_THRESHOLD,
    add_eos: bool = ADD_EOS,
) -> Optional[Tuple]:
    """
    Worker function executed inside a separate process.

    Responsibilities:
      - Create a tokenizer instance local to the worker process.
      - Filter rows for English according to `lang_mode`.
      - Tokenize each kept row and optionally append EOS.
      - Save the resulting token id sequence as a compact raw binary
        chunk file (atomic write).

    Returns
    -------
    Optional[tuple]
      If PROFILE is enabled, returns a long tuple with timing stats and
      a sample for verification; otherwise returns a short tuple. Returns
      None on error or if no tokens were produced.
    """
    # create tokenizer in worker (avoid pickling heavy tokenizer across processes)
    try:
        try:
            enc = tiktoken.get_encoding(tokenizer_model)
        except Exception:
            enc = tiktoken.encoding_for_model(tokenizer_model)
    except Exception as e:
        logger.exception("Worker failed to create tokenizer: %s", e)
        return None

    eos_id = _get_eos_token_id(enc) if add_eos else None
    if add_eos and eos_id is None:
        logger.warning("ADD_EOS requested but EOS token id couldn't be determined; EOS will be skipped.")

    # timers for profiling
    t_chunk_start = perf_counter()
    lang_time = 0.0
    token_time = 0.0
    save_time = 0.0

    # 1) language filtering per-row (in-worker to parallelize cost)
    num_input_rows = 0
    kept_rows: List[str] = []
    for txt in chunk_texts:
        num_input_rows += 1
        if not txt:
            continue
        t0 = perf_counter()
        keep = is_english_text(txt, mode=lang_mode, ascii_threshold=ascii_threshold)
        lang_time += perf_counter() - t0
        if keep:
            kept_rows.append(txt)

    num_kept_rows = len(kept_rows)
    if num_kept_rows == 0:
        # nothing to tokenize/save
        return None

    # 2) tokenization (per-row to avoid constructing monstrous temporary lists)
    token_ids: List[int] = []
    for row in kept_rows:
        t0 = perf_counter()
        try:
            toks = enc.encode_ordinary(row)
        except Exception:
            toks = enc.encode(row)
        token_time += perf_counter() - t0
        if toks:
            token_ids.extend(toks)
            if eos_id is not None:
                token_ids.append(eos_id)

    if not token_ids:
        return None

    # 3) choose compact dtype
    MAX_UINT16 = np.iinfo(np.uint16).max
    if max(token_ids) > MAX_UINT16:
        dtype = np.uint32
    else:
        dtype = np.uint16

    arr = np.array(token_ids, dtype=dtype)
    out_path = Path(out_dir) / f"chunk_{chunk_id:06d}.bin"

    # 4) atomic write
    t_save_start = perf_counter()
    if out_path.exists():
        save_time = 0.0
        logger.debug("Chunk %s exists, skipping write", out_path.name)
    else:
        try:
            data_bytes = arr.tobytes()
            _safe_atomic_write_raw_bytes(data_bytes, out_path)
            save_time = perf_counter() - t_save_start
        except Exception as e:
            logger.exception("Failed to save chunk %d -> %s : %s", chunk_id, out_path, e)
            return None

    # 5) prepare verification sample (small) and return profiling info if requested
    sample_tokens = arr[:sample_n].tolist()
    try:
        sample_text = enc.decode(sample_tokens)
    except Exception:
        sample_text = " ".join(str(t) for t in sample_tokens)

    total_elapsed = perf_counter() - t_chunk_start

    if PROFILE:
        return (
            str(out_path),
            int(arr.size),
            sample_tokens,
            sample_text,
            float(lang_time),
            float(token_time),
            float(save_time),
            int(num_input_rows),
            int(num_kept_rows),
            int(chunk_id),
            float(total_elapsed),
        )
    else:
        return str(out_path), int(arr.size), sample_tokens, sample_text


# ----------------- Merge: raw .bin -> tokens_merged.bin + index.json -----------------

def merge_chunks_to_raw_bin(chunk_files: list, out_bin: Path, out_index: Path, block_tokens: int = MERGE_BLOCK_TOKENS):
    """
    Stream-concatenate chunk raw `.bin` files into a single merged binary
    suitable for memory-mapped reads during training. Also write a small
    JSON index describing the output dtype and total length.

    Implementation notes
    --------------------
    - We scan each chunk as uint16 to measure maximum token id and length.
      This is a pragmatic assumption because worker writes prefer uint16.
    - After computing total length and output dtype, we allocate a memmap
      for the merged output and copy each chunk in small blocks to bound
      peak RAM usage.
    - Casting is performed on-the-fly so mixed chunk dtypes are tolerated
      (e.g., rare uint32 chunks will be cast to uint32 output if needed).
    """
    if not chunk_files:
        raise ValueError("No chunk files to merge")

    total = 0
    max_id = 0

    # first pass: fast uint16 scan to estimate max id and total tokens
    for f in chunk_files:
        size = Path(f).stat().st_size
        if size == 0:
            continue
        # interpret as uint16 for speed; if the chunk was uint32 this still
        # gives a usable element count (but value min/max may be misleading
        # only if values exceed uint16 which we detect later)
        src_u16 = np.memmap(str(f), dtype=np.uint16, mode='r')
        n = src_u16.shape[0]
        if n > 0:
            local_max = int(src_u16.max())
            if local_max > max_id:
                max_id = local_max
            total += n
        del src_u16
        gc.collect()

    # choose output dtype conservatively based on discovered max token id
    if max_id <= np.iinfo(np.uint16).max:
        out_dtype = np.uint16
    elif max_id <= np.iinfo(np.uint32).max:
        out_dtype = np.uint32
    else:
        out_dtype = np.int64

    logger.info("Merging %d files -> %s (dtype=%s)", len(chunk_files), out_bin, out_dtype)

    out_bin.parent.mkdir(parents=True, exist_ok=True)
    merged = np.memmap(str(out_bin), dtype=out_dtype, mode='w+', shape=(total,))

    cursor = 0
    for idx, f in enumerate(chunk_files):
        logger.info("Merging chunk %d/%d: %s", idx+1, len(chunk_files), f)
        # read as uint16 and cast to output dtype per-block; this is cheap
        # and avoids loading full chunk to RAM.
        src = np.memmap(str(f), dtype=np.uint16, mode='r')
        n = src.shape[0]
        start = 0
        while start < n:
            end = min(start + block_tokens, n)
            merged[cursor: cursor + (end - start)] = src[start:end].astype(out_dtype, copy=False)
            cursor += (end - start)
            start = end
        del src
        gc.collect()

    merged.flush()
    del merged
    gc.collect()

    # write compact index for consumer (training script)
    meta = {"dtype": str(out_dtype), "total_tokens": int(total), "file": str(out_bin)}
    with open(out_index, "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2)

    logger.info("Merged -> %s ; index -> %s", out_bin, out_index)


# ----------------- Orchestration (main entry point) -----------------

def run_pipeline() -> None:
    """
    Orchestrate the entire pipeline end-to-end.

    Steps:
      1) salvage stale temp files
      2) stream parquet rows into chunks
      3) dispatch tokenization jobs to worker processes
      4) collect profiling info and produced chunk files
      5) merge chunk files into final binary + index

    The function logs progress and profiling information to the configured
    logger and writes a CSV profiling file if PROFILE is True.
    """
    logger.info("Pipeline start: INPUT=%s OUTPUT=%s N_JOBS=%d", INPUT_DIR, OUTPUT_DIR, N_JOBS)
    salvage_stale_temp_files(OUTPUT_DIR)

    chunk_gen = iter_raw_text_chunks_from_parquets(INPUT_DIR, PARQUET_GLOB, batch_rows=BATCH_ROWS, target_chunk_bytes=TARGET_CHUNK_BYTES)

    saved_files: List[str] = []
    total_tokens = 0

    # profiling aggregates
    total_lang_time = 0.0
    total_token_time = 0.0
    total_save_time = 0.0
    total_chunks_profiled = 0
    total_input_rows = 0
    total_kept_rows = 0

    max_pending = max(1, N_JOBS * PREFETCH_FACTOR)

    csv_file = None
    csv_writer = None
    if PROFILE:
        write_header = not PROFILE_CSV.exists()
        csv_file = open(PROFILE_CSV, "a", newline="", encoding="utf-8")
        csv_writer = csv.writer(csv_file)
        if write_header:
            csv_writer.writerow(["chunk_id", "num_input_rows", "num_kept_rows", "token_count", "lang_time_s", "token_time_s", "save_time_s", "total_time_s"])
            csv_file.flush()

    try:
        while True:
            to_dispatch: List[Tuple[List[str], int]] = []
            for _ in range(max_pending):
                try:
                    chunk_texts, chunk_id = next(chunk_gen)
                    to_dispatch.append((chunk_texts, chunk_id))
                except StopIteration:
                    break
            if not to_dispatch:
                break

            logger.info("Dispatching %d chunks to workers (n_jobs=%d)", len(to_dispatch), N_JOBS)
            with parallel_backend("loky"):
                results = Parallel(n_jobs=N_JOBS, backend="loky", prefer="processes", verbose=PARALLEL_VERBOSE)(
                    delayed(tokenize_filter_and_save)(chunk_texts, chunk_id, str(OUTPUT_DIR), TOKENIZER_MODEL, 20, LANG_MODE, ASCII_THRESHOLD, ADD_EOS)
                    for chunk_texts, chunk_id in to_dispatch
                )

            for res in results:
                if res is None:
                    continue
                if PROFILE:
                    (
                        out_path, token_count, sample_tokens, sample_text,
                        lang_time, token_time, save_time,
                        num_input_rows, num_kept_rows, chunk_id, total_elapsed
                    ) = res
                    csv_writer.writerow([int(chunk_id), int(num_input_rows), int(num_kept_rows), int(token_count), f"{lang_time:.6f}", f"{token_time:.6f}", f"{save_time:.6f}", f"{total_elapsed:.6f}"])
                    csv_file.flush()
                    total_lang_time += lang_time
                    total_token_time += token_time
                    total_save_time += save_time
                    total_chunks_profiled += 1
                    total_input_rows += num_input_rows
                    total_kept_rows += num_kept_rows
                    total_tokens += token_count
                else:
                    out_path, token_count, sample_tokens, sample_text = res
                    total_tokens += token_count

                saved_files.append(out_path)
                short = sample_text if len(sample_text) <= 200 else sample_text[:200] + "…[truncated]"
                logger.info("Chunk saved: %s — tokens=%d — sample tokens=%s", out_path, token_count, sample_tokens)
                logger.info("Chunk decoded sample -> %s", repr(short))
            gc.collect()

        # fallback discovery if saved_files is empty (e.g., resumed run)
        if not saved_files:
            saved_files = [str(p) for p in OUTPUT_DIR.glob("chunk_*.bin")]
        if not saved_files:
            logger.info("No chunk files found/produced.")
            return

        # profiling summary
        if PROFILE and total_chunks_profiled > 0:
            logger.info("=== Profiling summary ===")
            logger.info("Chunks profiled: %d", total_chunks_profiled)
            logger.info("Total input rows: %d, total kept rows: %d, total tokens: %d", total_input_rows, total_kept_rows, total_tokens)
            logger.info("Total lang detection time: %.3fs", total_lang_time)
            logger.info("Total tokenization time: %.3fs", total_token_time)
            logger.info("Total save time: %.3fs", total_save_time)
            total_work = total_lang_time + total_token_time + total_save_time
            if total_work > 0:
                logger.info("Lang detection: %.2f%% ; Tokenization: %.2f%% ; Save: %.2f%%", 100.0 * total_lang_time / total_work, 100.0 * total_token_time / total_work, 100.0 * total_save_time / total_work)
            logger.info("Avg lang time / chunk: %.4fs", total_lang_time / total_chunks_profiled)
            logger.info("Avg token time / chunk: %.4fs", total_token_time / total_chunks_profiled)
            if total_token_time > 0:
                logger.info("Approx tokens/sec during tokenization: %.1f tok/s", total_tokens / total_token_time)

        # numeric sort by chunk id extracted from filename and then merge
        saved_files.sort(key=lambda f: int(re.sub(r"\D", "", Path(f).stem)))
        logger.info("Produced %d chunk files (approx total tokens=%d). Starting merge...", len(saved_files), total_tokens)

        merge_chunks_to_raw_bin(saved_files, MERGED_BIN, MERGED_INDEX, block_tokens=MERGE_BLOCK_TOKENS)
        logger.info("Final merged binary at: %s", MERGED_BIN)
        logger.info("Final merged index at: %s", MERGED_INDEX)

    except Exception:
        logger.exception("Pipeline failed")
        raise
    finally:
        if csv_file:
            try:
                csv_file.close()
            except Exception:
                pass


if __name__ == "__main__":
    t0 = time.time()
    run_pipeline()
    logger.info("All done in %.1fs", time.time() - t0)

    try:
        import winsound
        duration = 100
        freq = 200
        while freq < 10000:
            winsound.Beep(freq, duration)
            freq = int(freq * 1.1) + 1
    except Exception:
        pass


19:13:19 [INFO] Starting run; writing logs to logs\extract_1756141999.log
19:13:19 [INFO] Pipeline start: INPUT=datasets/4GB OUTPUT=outputs\extract_v12 N_JOBS=16
19:13:19 [INFO] Using string column 'text' for text extraction
19:13:26 [INFO] Dispatching 64 chunks to workers (n_jobs=16)
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  46 out of  64 | elapsed:   32.2s remaining:   12.5s
[Parallel(n_jobs=16)]: Done  59 out of  64 | elapsed:   40.1s remaining:    3.3s
[Parallel(n_jobs=16)]: Done  64 out of  64 | elapsed:   41.8s finished
19:14:08 [INFO] Chunk saved: outputs\extract_v12\chunk_000000.bin — tokens=10850108 — sample tokens=[464, 13362, 12091, 198, 1890, 477, 262, 1842, 11, 19661, 290, 10731, 287, 12091, 2517, 268, 447, 247, 82, 3835]
19:14:08 [INFO] Chunk decoded sample -> 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books'
19:14:08 [INFO] Chunk saved: outputs\extract_v12\chunk_000001.bin 

In [2]:
try:
    import winsound
    duration = 100  # milliseconds
    freq = 200  # Hz
    while freq<10000:
        winsound.Beep(freq, duration)
        freq = int(freq * 1.1) + 1
except:
    pass
